In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from datetime import datetime
import pytz
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
apps_df = pd.read_csv('Play_Store_Data.csv')
reviews_df = pd.read_csv('User_Reviews.csv')

In [ ]:
required_cols = ['App', 'Category', 'Rating', 'Reviews', 'Installs', 'Size']
apps_df.dropna(subset=required_cols, inplace=True)

In [ ]:
apps_df['Rating'] = pd.to_numeric(apps_df['Rating'], errors='coerce')
apps_df.dropna(subset=['Rating'], inplace=True)
apps_df = apps_df[apps_df['Rating'] <= 5]

In [ ]:
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')
apps_df.dropna(subset=['Reviews'], inplace=True)
apps_df['Reviews'] = apps_df['Reviews'].astype(int)

In [ ]:
apps_df['Installs'] = (apps_df['Installs'].astype(str)
                       .str.replace(',', '', regex=False)
                       .str.replace('+', '', regex=False)
                       .str.strip())
apps_df['Installs'] = pd.to_numeric(apps_df['Installs'], errors='coerce')
apps_df.dropna(subset=['Installs'], inplace=True)
apps_df['Installs'] = apps_df['Installs'].astype(int)
print("Sample Installs:", apps_df['Installs'].head().tolist())

In [ ]:
def convert_size(size):
    size = str(size).strip()
    try:
        if size.endswith('M'):
            return float(size[:-1])
        elif size.endswith('k'):
            return float(size[:-1]) / 1024
    except:
        pass
    return np.nan

apps_df['Size'] = apps_df['Size'].apply(convert_size)
apps_df.dropna(subset=['Size'], inplace=True)
print(f"Rows after Size cleaning: {len(apps_df)}")

In [ ]:
reviews_df['Sentiment_Subjectivity'] = pd.to_numeric(reviews_df['Sentiment_Subjectivity'], errors='coerce')
reviews_df.dropna(subset=['Sentiment_Subjectivity'], inplace=True)

avg_subjectivity = reviews_df.groupby('App')['Sentiment_Subjectivity'].mean().reset_index()
avg_subjectivity.columns = ['App', 'Avg_Subjectivity']

merged_df = apps_df.merge(avg_subjectivity, on='App', how='inner')
print(f"Rows after merge: {len(merged_df)}")

In [ ]:
allowed_categories = ['GAME', 'BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION',
                      'DATING', 'ENTERTAINMENT', 'SOCIAL', 'EVENTS']

merged_df = merged_df[merged_df['Rating'] > 3.5]
merged_df = merged_df[merged_df['Reviews'] > 500]
merged_df = merged_df[merged_df['Installs'] > 50000]
merged_df = merged_df[merged_df['Avg_Subjectivity'] > 0.5]
merged_df = merged_df[~merged_df['App'].str.contains('s', case=False, na=False)]
merged_df = merged_df[merged_df['Category'].isin(allowed_categories)]
print(f"Rows after all filters: {len(merged_df)}")

In [ ]:
category_label_map = {
    'BEAUTY': 'सौंदर्य',
    'BUSINESS': 'வணிகம்',
    'DATING': 'Dating'
}
merged_df['Category_Label'] = merged_df['Category'].map(category_label_map).fillna(merged_df['Category'])

In [ ]:
ist = pytz.timezone('Asia/Kolkata')
now_ist = datetime.now(ist)
hour_ist = now_ist.hour
minute_ist = now_ist.minute
current_decimal = hour_ist + minute_ist / 60
window_start = 17.0
window_end = 19.0
in_window = window_start <= current_decimal <= window_end
print(f"Current IST: {now_ist.strftime('%I:%M %p')} | Chart visible: {in_window}")

In [ ]:
html_output_path = "./"
os.makedirs(html_output_path, exist_ok=True)

if merged_df.empty:
    chart_html = "<p style='color:white;text-align:center;padding:40px;'>No data available for the selected filtering criteria.</p>"

elif not in_window:
    chart_html = ""

else:
    color_map = {'GAME': 'pink'}
    default_colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
                      '#19D3F3', '#FF6692', '#B6E880']
    non_game_cats = [c for c in merged_df['Category'].unique() if c != 'GAME']
    color_idx = 0
    for cat in non_game_cats:
        color_map[cat] = default_colors[color_idx % len(default_colors)]
        color_idx += 1

    fig = go.Figure()

    for cat in merged_df['Category'].unique():
        cat_data = merged_df[merged_df['Category'] == cat]
        label = category_label_map.get(cat, cat)
        fig.add_trace(go.Scatter(
            x=cat_data['Size'],
            y=cat_data['Rating'],
            mode='markers',
            name=label,
            marker=dict(
                size=np.sqrt(cat_data['Installs']) / 80,
                color=color_map.get(cat, '#636EFA'),
                opacity=0.7,
                line=dict(width=0.5, color='white')
            ),
            customdata=np.stack([
                cat_data['App'],
                cat_data['Category_Label'],
                cat_data['Rating'],
                cat_data['Size'],
                cat_data['Installs'],
                cat_data['Reviews']
            ], axis=-1),
            hovertemplate=(
                "<b>App:</b> %{customdata[0]}<br>"
                "<b>Category:</b> %{customdata[1]}<br>"
                "<b>Rating:</b> %{customdata[2]}<br>"
                "<b>Size (MB):</b> %{customdata[3]}<br>"
                "<b>Installs:</b> %{customdata[4]:,}<br>"
                "<b>Reviews:</b> %{customdata[5]:,}<extra></extra>"
            )
        ))

    fig.update_layout(
        title=dict(
            text='App Size vs Average Rating (Bubble Size = Installs)',
            font=dict(size=16, color='white')
        ),
        xaxis=dict(
            title='App Size (MB)',
            title_font=dict(size=13, color='white'),
            tickfont=dict(color='white')
        ),
        yaxis=dict(
            title='Average Rating',
            title_font=dict(size=13, color='white'),
            tickfont=dict(color='white')
        ),
        legend=dict(font=dict(color='white'), bgcolor='rgba(0,0,0,0)'),
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        margin=dict(l=60, r=30, t=70, b=60),
        width=980,
        height=560
    )

    chart_html = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    fig.write_html(os.path.join(html_output_path, 'bubble_chart.html'), full_html=False, include_plotlyjs='inline')
    fig.show()
    print("Chart saved: bubble_chart.html")

In [ ]:
dashboard_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Bubble Chart Dashboard</title>
    <style>
        body {{ background: #0d0d0d; color: white; font-family: 'Segoe UI', sans-serif; margin: 0; padding: 20px; }}
        h1 {{ text-align: center; color: #a78bfa; margin-bottom: 8px; }}
        .subtitle {{ text-align: center; color: #9ca3af; font-size: 13px; margin-bottom: 30px; }}
        .chart-wrapper {{ display: {'block' if in_window and not merged_df.empty else 'none'}; }}
        .no-data {{ text-align: center; padding: 50px; color: #ef4444; font-size: 15px; display: {'block' if merged_df.empty else 'none'}; }}
        .time-notice {{ text-align:center; padding: 30px; color: #f59e0b; font-size: 14px; display: {'block' if not in_window else 'none'}; }}
    </style>
</head>
<body>
    <h1>Play Store Bubble Chart</h1>
    <div class="subtitle">Filtered: Rating &gt; 3.5 | Reviews &gt; 500 | Installs &gt; 50K | Subjectivity &gt; 0.5 | No 'S' in App Name</div>

    <div class="no-data">No data available for the selected filtering criteria.</div>

    <div class="time-notice">
        ⏰ This visualization is available only between <strong>5:00 PM – 7:00 PM IST</strong>.<br>
        Current IST: <strong>{now_ist.strftime('%I:%M %p')}</strong>
    </div>

    <div class="chart-wrapper">
        {chart_html}
    </div>
</body>
</html>
"""

dashboard_path = os.path.join(html_output_path, 'bubble_dashboard.html')
with open(dashboard_path, 'w', encoding='utf-8') as f:
    f.write(dashboard_html)

print(f"Dashboard saved: {dashboard_path}")
print(f"IST Time: {now_ist.strftime('%I:%M %p')} | Window active: {in_window}")